In [6]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix

# -------------------------
# 0) Daten laden
# -------------------------
df = pd.read_csv("survey_results_cleaned_final.csv")
print("Loaded:", df.shape)

# Bool-Spalten ggf. in int umwandeln
bool_cols = df.select_dtypes(include=["bool"]).columns
for col in bool_cols:
    df[col] = df[col].astype(int)

target_col = "Employment"

# Target muss vorhanden sein
df = df.dropna(subset=[target_col]).copy()

# ✅ NEU: seltene/unnütze Klasse droppen (verhindert 1-sample Klassen im Test)
df = df[df[target_col] != "i prefer not to say"].copy()

# Optional: "Other" extrem selten -> droppen (bei dir waren es 2)
df = df[df[target_col] != "Other"].copy()

print("\nTarget distribution:")
print(df[target_col].value_counts())

# -------------------------
# 1) Feature-Spalten bestimmen
# -------------------------
text_cols = df.select_dtypes(include=["object"]).columns.tolist()
text_cols = [c for c in text_cols if c not in {"cluster", target_col}]  # target nicht in Text
df["__text__"] = df[text_cols].fillna("").agg(" ".join, axis=1)

num_cols = df.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
num_cols = [c for c in num_cols if c not in {target_col, "cluster"}]

# Optional: ID raus, falls vorhanden
for maybe_id in ["ResponseId"]:
    if maybe_id in num_cols:
        num_cols.remove(maybe_id)

print("\nTextspalten:", len(text_cols))
print("Numerische Spalten:", len(num_cols))

# ✅ NEU: Numerische NaNs droppen (oder imputer verwenden)
df = df.dropna(subset=num_cols).copy()

X = df[["__text__"] + num_cols].copy()
y = df[target_col].astype(str).copy()

print("\nX shape:", X.shape)
print("y shape:", y.shape)

# -------------------------
# 2) Train/Test Split
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTrain size:", len(X_train))
print("Test size:", len(X_test))
print("Train dist:\n", y_train.value_counts(normalize=True))
print("Test dist:\n", y_test.value_counts(normalize=True))

# -------------------------
# 3) Preprocessing
# -------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=5,            # ✅ NEU (vorher 2)
            max_df=0.9,
            max_features=20000,  # ✅ NEU
            sublinear_tf=True    # ✅ NEU
        ), "__text__"),
        ("num", StandardScaler(), num_cols)
    ],
    remainder="drop"
)

# -------------------------
# 4) Pipeline
# -------------------------
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("feature_selection", SelectFromModel(
        LinearSVC(penalty="l1", dual=False, C=0.5, max_iter=20000)
    )),
    ("classifier", LinearSVC(max_iter=20000))
])

# -------------------------
# 5) GridSearch
# -------------------------
parameters = {
    "preprocessing__text__ngram_range": [(1, 1), (1, 2)],
    "preprocessing__text__min_df": [5, 10],      # ✅ NEU: stabilere Varianten
    "preprocessing__text__max_df": [0.9],

    "classifier__C": [0.5, 1.0, 2.0],
    "classifier__class_weight": [None, "balanced"],
}

grid = GridSearchCV(
    pipeline,
    param_grid=parameters,
    scoring="f1_macro",
    verbose=2,
    cv=3,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("\nBeste Performance (CV, f1_macro):", grid.best_score_)
print("Beste Parameter:\n", grid.best_params_)

# -------------------------
# 6) Evaluation
# -------------------------
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("\nClassification Report (Test):")
print(classification_report(y_test, y_pred, zero_division=0))

print("Confusion Matrix (rows=true, cols=pred):")
labels_sorted = sorted(y.unique())
print(confusion_matrix(y_test, y_pred, labels=labels_sorted))

print("\nClassification Report (Train):")
y_pred_train = best_model.predict(X_train)
print(classification_report(y_train, y_pred_train, zero_division=0))


Loaded: (20603, 40)

Target distribution:
Employment
employed                                                16257
independent contractor, freelancer, or self-employed     2709
student                                                   995
not employed                                              594
retired                                                     2
Name: count, dtype: int64

Textspalten: 30
Numerische Spalten: 8

X shape: (12982, 9)
y shape: (12982,)

Train size: 10385
Test size: 2597
Train dist:
 Employment
employed                                                0.850650
independent contractor, freelancer, or self-employed    0.116418
not employed                                            0.018199
student                                                 0.014733
Name: proportion, dtype: float64
Test dist:
 Employment
employed                                                0.850597
independent contractor, freelancer, or self-employed    0.116673
not employed                

KeyboardInterrupt: 

In [6]:
# XGBoost + OneHot-Encoding – JobSat (Low/Medium/High)
# Ziel: JobSat klassifizieren (3 Klassen)
# Features: Numerik + kategoriale Spalten via OneHotEncoder
# Modell: XGBClassifier

# %%
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

from xgboost import XGBClassifier

# %%
# 1) Daten laden
df = pd.read_csv("survey_results_cleaned_final.csv")
print("Loaded:", df.shape)

# Optional: bool -> int (falls vorhanden)
bool_cols = df.select_dtypes(include=["bool"]).columns
for c in bool_cols:
    df[c] = df[c].astype(int)

# %%
# 2) Target: JobSat -> Klassen (Low/Medium/High)
target_col = "JobSat"

# Nur Zeilen behalten, wo JobSat vorhanden ist
df = df.dropna(subset=[target_col]).copy()

def map_jobsat(x):
    x = float(x)
    if x <= 3:
        return 0   # Low
    elif x <= 6:
        return 1   # Medium
    else:
        return 2   # High

y = df[target_col].apply(map_jobsat).astype(int)

print("Target distribution:")
print(y.value_counts().sort_index())

# %%
# 3) Feature-Spalten bestimmen
# -> Numerik: int/float (ohne JobSat, ohne cluster falls vorhanden)
# -> Kategorial: object (Strings), ebenfalls ohne cluster

drop_cols = {target_col}
if "cluster" in df.columns:
    drop_cols.add("cluster")

# Numerische Spalten
num_cols = df.select_dtypes(include=["int64","float64","int32","float32"]).columns.tolist()
num_cols = [c for c in num_cols if c not in drop_cols]

# Kategoriale Spalten
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
cat_cols = [c for c in cat_cols if c not in drop_cols]

# Optional: ID-Spalten raus (falls vorhanden)
for maybe_id in ["ResponseId"]:
    if maybe_id in num_cols: num_cols.remove(maybe_id)
    if maybe_id in cat_cols: cat_cols.remove(maybe_id)

print("Numeric cols:", len(num_cols))
print("Categorical cols:", len(cat_cols))

# %%
# 4) Gemeinsamer Clean-Step: nur Reihen behalten, wo Features + Target vollständig sind
# Für OneHot ist NaN ok (Encoder kann mit fehlenden umgehen, wenn wir sie als Kategorie behandeln),
# ABER XGBoost + sklearn Pipeline ist am stabilsten, wenn wir NaNs in cat als "MISSING" füllen.
X = df[num_cols + cat_cols].copy()

# Kategoriale NaNs füllen
for c in cat_cols:
    X[c] = X[c].fillna("MISSING").astype(str)

# Numerische NaNs: XGBoost kann NaNs grundsätzlich, aber damit train/test konsistent ist,
# lassen wir sie drin. Wenn du willst, kannst du hier auch dropna machen:
# X = X.dropna(subset=num_cols)
# y = y.loc[X.index]

# Wichtig: Index syncen
y = y.loc[X.index]
assert len(X) == len(y), f"Mismatch X={len(X)} vs y={len(y)}"

print("X shape:", X.shape, "| y shape:", y.shape)

# %%
# 5) Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

# %%
# 6) Preprocessing: OneHotEncoder für Kategorien
#    Wichtig: seltene Kategorien bündeln -> verhindert Feature-Explosion.
#    min_frequency= z.B. 50 heißt: Kategorien die <50 mal vorkommen, werden "infrequent".
#    (Wenn sklearn zu alt ist und min_frequency nicht unterstützt -> sag Bescheid, ich gebe Fallback.)
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=50,               # <<< ggf. anpassen (25 / 50 / 100)
            sparse_output=True
        ), cat_cols),
        ("num", "passthrough", num_cols),
    ],
    remainder="drop"
)

# %%
# 7) Modell: XGBoost Multiclass
xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)

model = Pipeline([
    ("preprocess", preprocessor),
    ("clf", xgb)
])

# %%
# 8) Trainieren
model.fit(X_train, y_train)

# %%
# 9) Evaluation
y_pred = model.predict(X_test)

print("Classification Report (Test):")
print(classification_report(y_test, y_pred, labels=[0,1,2], target_names=["Low","Medium","High"]))

print("Confusion Matrix (rows=true, cols=pred):")
print(confusion_matrix(y_test, y_pred, labels=[0,1,2]))

# Optional: Train-Report (Overfitting-Check)
y_pred_train = model.predict(X_train)
print("\nClassification Report (Train):")
print(classification_report(y_train, y_pred_train, labels=[0,1,2], target_names=["Low","Medium","High"]))


Loaded: (19826, 38)
Target distribution:
JobSat
0     1022
1     3842
2    12351
Name: count, dtype: int64
Numeric cols: 6
Categorical cols: 30
X shape: (17215, 36) | y shape: (17215,)
Train size: 13772
Test size: 3443
Classification Report (Test):
              precision    recall  f1-score   support

         Low       0.33      0.06      0.11       204
      Medium       0.38      0.12      0.18       769
        High       0.75      0.96      0.84      2470

    accuracy                           0.72      3443
   macro avg       0.49      0.38      0.38      3443
weighted avg       0.64      0.72      0.65      3443

Confusion Matrix (rows=true, cols=pred):
[[  13   50  141]
 [  18   93  658]
 [   8  103 2359]]

Classification Report (Train):
              precision    recall  f1-score   support

         Low       0.99      0.41      0.58       818
      Medium       0.92      0.38      0.54      3073
        High       0.81      0.99      0.89      9881

    accuracy            

# Mindestanforderungen Klassifikation
- Führen Sie mit dem Algorithmus Ihrer Wahl eine Klassifikationsaufgabe auf Ihren Daten durch.
    - Ziel war die Vorhersage der Zielvariable `Employment`
    - Klassen waren _employed_, _self-employed_, _student_ und _not-employed_
    - sehr seltene Klassen wurden entfernt
    - Wir haben die **lineare Support Vector Machine** genutzt
    - Einsatz in der Pipeline mit Text- und numerischen Features
___
- Teilen Sie dazu zunächst die Daten auf, um Overfitting beim Trainieren des Algorithmus und bei der Parameterauswahl zu vermeiden. Erklären Sie die gewählte Strategie und die Größenverhältnisse.
    - Wir haben auf 80/20 gesplittet, also 80% Train und 20% Test (`train_test_split(test_size=0.2)`)
    - wir haben `stratify=y` angewandt um eine ähnliche Klassenverteilung in den Train- und Testdaten zu erhalten
    - um Overfitting bei Parametern zu verhindern haben wir Parameter-Tuning nur auf das Trainingsset mit 3-facher Cross-Validation angewandt
---
- Wählen Sie geeignete Features aus und setzen Sie die Parameter des Algorithmus. Beschreiben Sie das gewälhte Vorgehen für die Auswahl der Features und Parameter. Berichten Sie den Parameterraum und die final gewählten Parameter. Geben Sie die Performanz auf den Trainingsdaten (bzw. Entwicklungsdaten, falls verwendet) an.
    - **Features**:
        - alle Textspalten zu `__text__` zusammengefasst, TfidfVectorizer
        - bei numerischen Spalten wurden fehlende Werte mit dem Median aufgefüllt und anschließend Standardisiert
        - Entfernt wurden: `ResponseId`, `Age` (als String) und `ConvertedCompTotal` (Gehälter)
    - **Parameterauswahl**
        - GridSearchCV (cv=3) auf Trainingsdaten
        - Optimierungsmaß: macro-F1
    - **Parameterraum**:
        - `ngram_range`: (1,1), (1,2)
        - `min_df`: 5, 10
        - `max_df`: 0.9
        - `C`: 0.5, 1.0, 2.0
        - `class_weight`: None, balanced
    - **Beste Parameter**:
        - `ngram_range` = (1,2)
        - `min_df` = 10
        - `C` = 0.5
        - `class_weight` = balanced
    - **Train/Dev-Leistung**
        - CV macro-F1 ~ 0.75
---
- Evaluieren Sie die Klassifikation auf den ungesehenen Testdaten. Betrachten Sie Precision und Recall sowie den F-Wert. Welches Maß ist für Ihre Anwendung wichtiger? Bewerten Sie Ihr Ergebnis. Ist es in der Praxis voraussichtlich zufriedenstellend?
    - Test-Ergebnisse
        - Accuracy: 0.93
        - macro-F1: 0.77
    - Precision/Recall
        - sehr hoch für _employed_
        - gut für _self-employed_
        - geringer für _student_ und not _employed_ (kleine Klassen)
    - wichtigstes Maß
        - macro-F1 ist wichtiger als die Accuracy, da die Klassen sehr stark unbalanciert sind
    - Bewertung
        - das Modell erkennt Mehrheitsklassen sehr zuverlässig
        - Minderheitsklassen werden schwieriger erkannt, hier wären in der Praxis voraussichtlich weitere Maßnahmen nötig
---
Weiter unten finden Sie noch weitere Lösungsansätze, unter anderem mit XG-Boost

# Codeerklärungen
- imports laden
- csv einlesen
- Zielvariable `Employment` festlegen, die durch Klassifikation vorhergesagt werden soll

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix

df = pd.read_csv("survey_results_cleaned_final.csv")
target_col = "Employment"

- Zeilen ohne `Employment` werden entfernt
- sehr seltene Klassen werden entfernt, da sie irrelevant sind und zu wenige Beispiele haben (`prefer not to say` & `Other`
- zudem wird `ResponseId` entfernt, da es nur eine ID ist - kein inhaltlicher Mehrwert
- `Age` ist redundant aufgrund von `AgeNum`
- `ConvertedCompTotal` zu viele leere Zellen/NaN-Werte

In [ ]:
df = df.dropna(subset=[target_col]).copy()
df = df[df[target_col] != "i prefer not to say"].copy()
df = df[df[target_col] != "Other"].copy()

df = df.drop(columns=["ResponseId", "Age", "ConvertedCompTotal"], errors="ignore")

- Nach `object`-Spalten suchen, da sie meist Text/Kategorien sind
- `Employment` muss entfernt werden, da sonst geschummelt werden würde
- Aus allen Textspalten eine gemeinsamen Text pro Zeile (`__text__`)
#### Warum?
- kategoriale Spalten als Textklassifikation zu behandeln
- Danach kann TFIDF Vectorizer wie beim Clustering numerische Features daraus machen

In [ ]:
text_cols = df.select_dtypes(include=["object"]).columns.tolist()
text_cols = [c for c in text_cols if c not in {target_col, "cluster"}]
df["__text__"] = df[text_cols].fillna("").agg(" ".join, axis=1)

- numerische Spalten sammeln
- Auch hier wird Zielvariable `Employment` ausgeschlossen

In [ ]:
num_cols = df.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
num_cols = [c for c in num_cols if c not in {target_col, "cluster"}]

- `x` enthält Features (Text und Numerisch)
- `y` enthält die Labels, also die Employment-Klassen
- Ausgabe der Shapes und Klassenverteilung um Werte zu überprüfen

In [ ]:
X = df[["__text__"] + num_cols].copy()
y = df[target_col].astype(str).copy()

print("X shape:", X.shape, "| y shape:", y.shape)
print("Target distribution:\n", y.value_counts())

- wir splitten in 80% Training und 20% Test
- `stratify=y` -> Klassenverteilung bleibt bei Train und Test ungefähr gleich
- stratify wichtig, weil Zielvariable stark unbalanciert ist

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

### Textspalten
- `TfidfVectorizer` macht aus Wörtern Zahlen
- dabei werden Wörter so gewichtet, dass häufige "Standardwörter" weiger wichtig sind und informative Wörter stärker zählen
- `max_features=20000` begrenzt die Anzahl der Textfeatures, damit die Werte nicht explodieren
- `sublinear_tf=True` dämpft extrem häufige Wörte zusätzlich

### Numerische Spalten
- `SimpleImputer(median)` füllt fehlende numerische Werte, also `NaN-Werte`, mit dem Median der Spalte
- StandardScaler skaliert die Zahlen auf vergleichbare Größenordnungen, was Support Vector Machines hilft, da sie empfindlich auf unterschiedliche Skalen reagieren können

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(sublinear_tf=True, max_features=20000), "__text__"),
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), num_cols)
    ],
    remainder="drop"
)

- in der Pipeline kommt nun die Vorverarbeitung und der Klassifikatior (LinearSVC) zusammen
- Warum LinearSVC?
    - für Textdaten mit vielen Features funktioniert lineare SVC oft sehr gut

In [ ]:
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LinearSVC(max_iter=20000))
])

- hier werden mehrere sinnvolle Einstellungen getestet:
- `ngram_range`:
    - (1,1) = nur einzelne Wörter
    - (1,2) = Wörter + Wortpaare -> Kontext besser zu erfassen
- `min_df`:
    - ignoriert sehr seltene Wörter (erzeugt oft nur rauschen)
- `C`:
    - Regularisierung der SVM: kleiner C entspricht stärkerer Regularisierung, ein größeres C bedeutet mehr Flexibilität
- `class_weight="balanced"`:
    - wichtig bei unbalancierten Klassen, da so kleine Klassen stärker gewichtet werden

In [ ]:
param_grid = {
    "preprocessing__text__ngram_range": [(1, 1), (1, 2)],
    "preprocessing__text__min_df": [5, 10],
    "preprocessing__text__max_df": [0.9],
    "classifier__C": [0.5, 1.0, 2.0],
    "classifier__class_weight": [None, "balanced"],
}

- GridSearchCV testet alle Parameterkombinationen
- Bewertung: `f1_macro`
    - sinnvoll, weil jede Klasse gleich gewichtet wird
- cv=3 bedeutet 3-fache Cross-Validation auf dem Trainingsset
- Testset bleibt unangetastet -> fairer Vergleich

In [ ]:
grid = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)

print("\nBest CV macro-F1:", grid.best_score_)
print("Best params:", grid.best_params_)

- bestes Modell aus der GridSearch wird verwendet
- dann einmalige Evaluierung auf den Testdaten
- Confusion Matrix zeigt, welche Klassen miteinander verwechselt wurden
- Mehrheitsklasse (employed) oft sehr gut
- Minderheitsklassen häufiger als employed vorhergesagt

In [5]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("\nTest report:\n", classification_report(y_test, y_pred, zero_division=0))

labels_sorted = best_model.named_steps["classifier"].classes_
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred, labels=labels_sorted))

X shape: (19810, 7) | y shape: (19810,)
Target distribution:
 Employment
employed                                                16316
independent contractor, freelancer, or self-employed     2483
student                                                   659
not employed                                              352
Name: count, dtype: int64
Fitting 3 folds for each of 24 candidates, totalling 72 fits

Best CV macro-F1: 0.7450045659917125
Best params: {'classifier__C': 0.5, 'classifier__class_weight': 'balanced', 'preprocessing__text__max_df': 0.9, 'preprocessing__text__min_df': 10, 'preprocessing__text__ngram_range': (1, 2)}

Test report:
                                                       precision    recall  f1-score   support

                                            employed       0.96      0.98      0.97      3263
independent contractor, freelancer, or self-employed       0.85      0.77      0.81       497
                                        not employed       0.71  

---
Alternativer Lösungsansatz, den wir probiert haben
## Klassifikation mit allen Werten

`JobSat` als Zielwert


In [16]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix


## Daten laden

Im One-Hot-Encoded Datensatz sind die One-Hot-Encoded Spalten mit bool Werten. Um es etwas einfacher zu gestalten werden diese Werte hier in Integer-Werte (False -> 0; True -> 1) umgewandelt.

In [17]:
df = pd.read_csv("survey_results_cleaned_final.csv")

bool_cols = df.select_dtypes(include=['bool']).columns

for col in bool_cols:
    df[col] = df[col].astype(int)
df.dtypes

ResponseId                          int64
MainBranch                         object
Age                                object
MaxAge                            float64
AgeNum                            float64
EdLevel                            object
Employment                         object
WorkExp                           float64
LearnCodeAI                        object
YearsCode                         float64
DevType                            object
OrgSize                            object
ICorPM                             object
RemoteWork                         object
RemoteCategoryNum                 float64
Industry                           object
AIThreat                           object
NewRole                            object
Country                            object
LanguageChoice                     object
LanguageHaveWorkedWith             object
DatabaseChoice                     object
DatabaseHaveWorkedWith             object
PlatformChoice                    

## Zielvariable in Klassen einteilen

Dadurch haben wir mehr Trainingsdaten, als wenn wir `JobSat` von 0-10 als Klassen defonieren würden

Klassen
- **Low**: 0–3
- **Medium**: 4–6
- **High**: 7–10


In [18]:
# Nur Zeilen behalten, wo JobSat vorhanden ist
df = df.dropna(subset=["JobSat"]).copy()


def map_jobsat(x):
    x = float(x)
    if x <= 3:
        return "Low"
    elif x <= 6:
        return "Medium"
    else:
        return "High"


## Feature-Spalten bestimmen

- Textspalten: `object` (Strings)
- Numerische Spalten: `int/float`

In [19]:
# Textspalten (Strings)
text_cols = df.select_dtypes(include=["object"]).columns.tolist()

df["__text__"] = df[text_cols].fillna("").agg(" ".join, axis=1)

# Numerische Spalten
num_cols = df.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()

# Zielspalte aus numerischen Features entfernen (falls vorhanden)
num_cols = [c for c in num_cols if c != "JobSat"]  #?

df = df.dropna(subset=["__text__"] + num_cols + ["JobSat"])

y = df["JobSat"].apply(map_jobsat)

print("Textspalten:", text_cols)
print("Numerische Spalten:", num_cols)

# Feature-Matrix aus den ausgewählten Spalten
X = df[["__text__"] + num_cols].copy()

X.head()

Textspalten: ['MainBranch', 'Age', 'EdLevel', 'Employment', 'LearnCodeAI', 'DevType', 'OrgSize', 'ICorPM', 'RemoteWork', 'Industry', 'AIThreat', 'NewRole', 'Country', 'LanguageChoice', 'LanguageHaveWorkedWith', 'DatabaseChoice', 'DatabaseHaveWorkedWith', 'PlatformChoice', 'PlatformHaveWorkedWith', 'WebframeChoice', 'WebframeHaveWorkedWith', 'DevEnvsChoice', 'DevEnvsHaveWorkedWith', 'OfficeStackAsyncHaveWorkedWith', 'CommPlatformHaveWorkedWith', 'CommPlatformWantToWorkWith', 'AIModelsChoice', 'AIModelsHaveWorkedWith', 'AISelect', 'AIAgents', 'AIAgent_Uses']
Numerische Spalten: ['ResponseId', 'MaxAge', 'AgeNum', 'WorkExp', 'YearsCode', 'RemoteCategoryNum', 'RemoteMissing', 'ConvertedCompTotal']


,__text__,ResponseId,MaxAge,AgeNum,WorkExp,YearsCode,RemoteCategoryNum,RemoteMissing,ConvertedCompTotal
0,i am a developer by profession 25-34 years old...,1,34.0,29.0,8.0,14.0,0.00,0,61659.84
1,i am a developer by profession 25-34 years old...,2,34.0,29.0,2.0,10.0,0.25,0,105102.00
3,i am a developer by profession 35-44 years old...,4,44.0,39.0,4.0,5.0,0.00,0,36435.36
4,i am a developer by profession 35-44 years old...,5,44.0,39.0,21.0,22.0,0.50,1,60000.00
5,i am a developer by profession 45-54 years old...,6,54.0,49.0,15.0,20.0,0.50,1,120000.00


## Train/Test Split

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))
print("Train class distribution:\n", y_train.value_counts(normalize=True))
print("Test class distribution:\n", y_test.value_counts(normalize=True))

Train size: 10391
Test size: 2598
Train class distribution:
 JobSat
High      0.717159
Medium    0.220383
Low       0.062458
Name: proportion, dtype: float64
Test class distribution:
 JobSat
High      0.717090
Medium    0.220554
Low       0.062356
Name: proportion, dtype: float64


## Preprocessing für Text und numerische Spalten

In [21]:
preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(), "__text__"),
        ("num", StandardScaler(), num_cols)
    ],
    remainder="drop"
)

## Pipeline definieren

- Preprocessing
- Feature-Selektion (SelectFromModel mit L1-LinearSVC)
- Klassifikator (LinearSVC)

In [22]:
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("feature_selection", SelectFromModel(LinearSVC(penalty="l1", dual=False, C=0.5))),
    ("classifier", LinearSVC())
])

## GridSearch

In [23]:
parameters = {
    # TF-IDF: nur word, nur die zwei wichtigsten Varianten
    "preprocessing__text__analyzer": ["word"],
    "preprocessing__text__ngram_range": [(1, 1), (1, 2)],
    "preprocessing__text__max_df": [0.9],
    "preprocessing__text__min_df": [2],

    # Klassifikator: 2 sinnvolle Regularisierungen + optional balancing
    "classifier__C": [1.0, 2.0],
    "classifier__class_weight": [None, "balanced"],
}

grid = GridSearchCV(pipeline, param_grid=parameters, verbose=2, cv=3, n_jobs=-1)

## Grid Search + Beste Parameter


In [24]:
grid.fit(X_train, y_train)

print("Beste Performance:", grid.best_score_)
print("Beste Parameter:\n", grid.best_params_)

Fitting 3 folds for each of 8 candidates, totalling 24 fits


C:\Users\MoritzSchwarz\PycharmProjects\data-analytics-project\.venv\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Beste Performance: 0.7177359602902076
Beste Parameter:
 {'classifier__C': 2.0, 'classifier__class_weight': None, 'preprocessing__text__analyzer': 'word', 'preprocessing__text__max_df': 0.9, 'preprocessing__text__min_df': 2, 'preprocessing__text__ngram_range': (1, 1)}


## Evaluation auf Testdaten

In [25]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("Classification Report (Test):")
print(classification_report(y_test, y_pred))

print("Confusion Matrix (rows=true, cols=pred):")
print(confusion_matrix(y_test, y_pred, labels=["Low", "Medium", "High"]))

Classification Report (Test):
              precision    recall  f1-score   support

        High       0.74      0.97      0.84      1863
         Low       0.50      0.01      0.02       162
      Medium       0.46      0.13      0.21       573

    accuracy                           0.72      2598
   macro avg       0.57      0.37      0.36      2598
weighted avg       0.67      0.72      0.65      2598

Confusion Matrix (rows=true, cols=pred):
[[   2   32  128]
 [   0   76  497]
 [   2   57 1804]]


In [26]:
y_pred_train = best_model.predict(X_train)

print("Classification Report (Train):")
print(classification_report(y_train, y_pred_train))

Classification Report (Train):
              precision    recall  f1-score   support

        High       0.75      0.97      0.85      7452
         Low       0.74      0.04      0.07       649
      Medium       0.50      0.16      0.24      2290

    accuracy                           0.73     10391
   macro avg       0.66      0.39      0.38     10391
weighted avg       0.69      0.73      0.66     10391



## Alternativ Ansatz mit XG-Boost

### Klassifikation mit allen Werten

`JobSat` als Zielwert


In [1]:
import os, glob
print("cwd:", os.getcwd())
print("local xgboost candidates:", glob.glob("xgboost*"))

cwd: C:\Users\MoritzSchwarz\PycharmProjects\data-analytics-project\Abgaben
local xgboost candidates: []


In [2]:
import sys
!{sys.executable} -m pip -V
!{sys.executable} -m pip uninstall -y xgboost
!{sys.executable} -m pip install --no-cache-dir -U xgboost

pip 25.2 from C:\Users\MoritzSchwarz\PycharmProjects\data-analytics-project\.venv\Lib\site-packages\pip (python 3.13)

Found existing installation: xgboost 3.1.2
Uninstalling xgboost-3.1.2:
  Successfully uninstalled xgboost-3.1.2
   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   -- ------------------------------------- 3.9/72.0 MB 20.3 MB/s eta 0:00:04
   ---- ----------------------------------- 8.9/72.0 MB 23.0 MB/s eta 0:00:03
   -------- ------------------------------- 14.7/72.0 MB 24.1 MB/s eta 0:00:03
   ----------- ---------------------------- 20.2/72.0 MB 24.4 MB/s eta 0:00:03
   -------------- ------------------------- 26.2/72.0 MB 25.1 MB/s eta 0:00:02
   ----------------- ---------------------- 31.5/72.0 MB 25.2 MB/s eta 0:00:02
   -------------------- ------------------- 37.0/72.0 MB 25.1 MB/s eta 0:00:02
   ----------------------- ---------------- 42.7/72.0 MB 25.4 MB/s eta 0:00:02
   --------------------------- ------------ 49.0/72.0 MB 25.8 MB/s et


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix

## Daten laden

Im One-Hot-Encoded Datensatz sind die One-Hot-Encoded Spalten mit bool Werten. Um es etwas einfacher zu gestalten werden diese Werte hier in Integer-Werte (False -> 0; True -> 1) umgewandelt.

In [4]:
df = pd.read_csv("One-Hot-Encoded.csv")


bool_cols = df.select_dtypes(include=['bool']).columns

for col in bool_cols:
    df[col] = df[col].astype(int)
df.dtypes

MaxAge                                                            float64
AgeNum                                                            float64
WorkExp                                                           float64
YearsCode                                                         float64
RemoteCategoryNum                                                 float64
                                                                   ...   
AIAgents_no, i use ai exclusively in copilot/autocomplete mode      int64
AIAgents_yes, i use ai agents at work daily                         int64
AIAgents_yes, i use ai agents at work monthly or infrequently       int64
AIAgents_yes, i use ai agents at work weekly                        int64
AIAgents_nan                                                        int64
Length: 455, dtype: object

## Zielvariable in Klassen einteilen

Dadurch haben wir mehr Trainingsdaten, als wenn wir `JobSat` von 0-10 als Klassen defonieren würden

Klassen
- **Low**: 0–3
- **Medium**: 4–6
- **High**: 7–10

In [5]:
# Nur Zeilen behalten, wo JobSat vorhanden ist
df = df.dropna(subset=["JobSat"]).copy()

def map_jobsat(x):
    x = float(x)
    if x <= 3:
        return 0
    elif x <= 6:
        return 1
    else:
        return 2

## Feature-Spalten bestimmen

- Textspalten: `object` (Strings)
- Numerische Spalten: `int/float`

In [6]:
# Textspalten (Strings)
text_cols = df.select_dtypes(include=["object"]).columns.tolist()

df["__text__"] = df[text_cols].fillna("").agg(" ".join, axis=1)

# Numerische Spalten
num_cols = df.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()

# Zielspalte aus numerischen Features entfernen (falls vorhanden)
num_cols = [c for c in num_cols if c != "JobSat"] #?

df = df.dropna(subset=["__text__"] + num_cols + ["JobSat"])

y = df["JobSat"].apply(map_jobsat)

print("Textspalten:", text_cols)
print("Numerische Spalten:", num_cols)

# Feature-Matrix aus den ausgewählten Spalten
X = df[["__text__"] + num_cols].copy()

X.head()

Textspalten: ['LanguageHaveWorkedWith', 'LanguageWantToWorkWith', 'DatabaseHaveWorkedWith', 'DatabaseWantToWorkWith', 'PlatformHaveWorkedWith', 'PlatformWantToWorkWith', 'WebframeHaveWorkedWith', 'WebframeWantToWorkWith', 'DevEnvsHaveWorkedWith', 'DevEnvsWantToWorkWith', 'OfficeStackAsyncHaveWorkedWith', 'OfficeStackAsyncWantToWorkWith', 'AIModelsHaveWorkedWith', 'AIModelsWantToWorkWith', 'AIAgent_Uses']
Numerische Spalten: ['MaxAge', 'AgeNum', 'WorkExp', 'YearsCode', 'RemoteCategoryNum', 'CompTotal', 'ConvertedCompYearly', 'MainBranch_i am a developer by profession', 'MainBranch_i am learning to code', 'MainBranch_i am not primarily a developer, but i write code sometimes as part of my work/studies', 'MainBranch_i code primarily as a hobby', 'MainBranch_i used to be a developer by profession, but no longer am', 'MainBranch_i work with developers or my work supports developers but am not a developer by profession', 'MainBranch_nan', 'Age_18-24 years old', 'Age_25-34 years old', 'Age_35

,__text__,MaxAge,AgeNum,WorkExp,YearsCode,RemoteCategoryNum,CompTotal,ConvertedCompYearly,MainBranch_i am a developer by profession,MainBranch_i am learning to code,...,"AISelect_yes, i use ai tools monthly or infrequently","AISelect_yes, i use ai tools weekly",AISelect_nan,"AIAgents_no, and i don't plan to","AIAgents_no, but i plan to","AIAgents_no, i use ai exclusively in copilot/autocomplete mode","AIAgents_yes, i use ai agents at work daily","AIAgents_yes, i use ai agents at work monthly or infrequently","AIAgents_yes, i use ai agents at work weekly",AIAgents_nan
0,"['bash/shell (all shells)', 'dart', 'sql'] ['d...",34.0,29.0,8.0,14.0,0.00,52800.0,61256.0,1,0,...,1,0,0,0,0,0,0,1,0,0
1,"['java'] ['java', 'python', 'swift'] ['dynamod...",34.0,29.0,2.0,10.0,0.25,90000.0,104413.0,1,0,...,0,1,0,1,0,0,0,0,0,0
3,"['java', 'kotlin', 'sql'] ['java', 'kotlin'] [...",44.0,39.0,4.0,5.0,0.00,31200.0,36197.0,1,0,...,0,1,0,0,0,0,0,1,0,0
7,"['bash/shell (all shells)', 'html/css', 'javas...",44.0,39.0,22.0,30.0,0.00,72000.0,72000.0,1,0,...,0,0,0,0,1,0,0,0,0,0
8,"['java', 'python', 'scala'] ['scala'] ['amazon...",34.0,29.0,9.0,15.0,0.00,70000.0,70000.0,1,0,...,1,0,0,0,0,0,0,0,0,1


## Train/Test Split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))
print("Train class distribution:\n", y_train.value_counts(normalize=True))
print("Test class distribution:\n", y_test.value_counts(normalize=True))

Train size: 12896
Test size: 3225
Train class distribution:
 JobSat
2    0.714020
1    0.224721
0    0.061259
Name: proportion, dtype: float64
Test class distribution:
 JobSat
2    0.713798
1    0.224806
0    0.061395
Name: proportion, dtype: float64


## Preprocessing für Text und numerische Spalten

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(), "__text__"),
        ("num", StandardScaler(), num_cols)
    ],
    remainder="drop"
)

## Pipeline definieren

- Preprocessing
- Feature-Selektion (SelectFromModel mit XGBoost-Feature-Importances)
- Klassifikator (XGBClassifier)

In [9]:
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("feature_selection", SelectFromModel(XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        n_jobs=-1,
        random_state=42,
        eval_metric="mlogloss"
    ))),
    ("classifier", XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        n_jobs=-1,
        random_state=42,
        eval_metric="mlogloss"
    ))
])

## GridSearchCV

In [10]:
parameters = {
    # TF-IDF: nur word, nur die zwei wichtigsten Varianten
    "preprocessing__text__analyzer": ["word"],
    "preprocessing__text__ngram_range": [(1, 1), (1, 2)],
    "preprocessing__text__max_df": [0.9],
    "preprocessing__text__min_df": [2],

    # XGBoost: kompakter, sinnvoller Suchraum
    "classifier__n_estimators": [300, 500],
    "classifier__max_depth": [4, 6],
    "classifier__learning_rate": [0.05, 0.1],
    "classifier__subsample": [0.8],
    "classifier__colsample_bytree": [0.8],
    "classifier__reg_lambda": [1.0, 2.0],
}

grid = GridSearchCV(pipeline, param_grid=parameters, verbose=2, cv=3, n_jobs=-1)

## Grid Search + Beste Parameter

In [ ]:
grid.fit(X_train, y_train)

print("Beste Performance:", grid.best_score_)
print("Beste Parameter:\n", grid.best_params_)

## Abbruch nach 30 mins

Fitting 3 folds for each of 32 candidates, totalling 96 fits


## Evaluation auf Testdaten

In [ ]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("Classification Report (Test):")
print(classification_report(y_test, y_pred))

print("Confusion Matrix (rows=true, cols=pred):")
print(confusion_matrix(y_test, y_pred, labels=[0,1,2]))

In [ ]:
y_pred_train = best_model.predict(X_train)

print("Classification Report (Train):")
print(classification_report(y_train, y_pred_train))